# Section 8 – Lecture 2: Designing Reliable Agent Flows using LangGraph

## 🧠 What You’ll Learn
- How to design a multi-step AI agent flow using LangGraph
- Add nodes and transitions to build a complete workflow
- Understand the benefits of deterministic branching and control

## ✅ Step 1: Define a State Class for Multi-step Workflow

In [ ]:
from pathlib import Path
import sys

def _find_root():
    hints = [
        Path("data") / "udemy" / "courses" / "ai_agents_bootcamp" / "course_repo",
        Path("course_repo"),
    ]
    def ok(p: Path) -> bool:
        return (p / "src" / "llm.py").is_file() or (p / "Section_5_Autonomous_Workflows").is_dir()
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if ok(p):
            return p
        for h in hints:
            cand = p / h
            if ok(cand):
                return cand.resolve()
    raise FileNotFoundError("Open this notebook from the AIAgentsBootcamp folder, or set cwd to that repo.")

_ROOT = _find_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from src.llm import REPO_ROOT, get_agent, get_autogen_config, get_embeddings, get_llm
llm = get_llm()


In [ ]:
class WorkflowState:
    def __init__(self, data, step=0):
        self.data = data
        self.step = step

## ✅ Step 2: Create LangGraph Nodes (Functions)

In [ ]:
def preprocess(state):
    print("[1] Preprocessing input data...")
    state.data = state.data.strip().lower()
    return state

def analyze(state):
    print("[2] Performing data analysis...")
    state.data = f"Analysis of '{state.data}' complete."
    return state

def summarize(state):
    print("[3] Summarizing results...")
    state.data = f"Summary: {state.data}"
    return state

## ✅ Step 3: Build and Run the LangGraph Workflow

In [ ]:
from langgraph.graph import StateGraph

builder = StateGraph(state_type=WorkflowState)
builder.add_node("preprocess", preprocess)
builder.add_node("analyze", analyze)
builder.add_node("summarize", summarize)
builder.set_entry_point("preprocess")
builder.add_edge("preprocess", "analyze")
builder.add_edge("analyze", "summarize")

workflow_graph = builder.compile()

initial_state = WorkflowState(data="   This is a TEST   ")
final_state = workflow_graph.invoke(initial_state)
print("\n✅ Final Workflow Output:")
print(final_state.data)

## ✅ Summary
- You designed a step-by-step agent workflow
- Each node performed a specific action on the state
- The flow was deterministic and reliable using LangGraph